# adabmDCA: train, generate, and explore sequence models

This notebook provides a guided interface to the high-level **adabmDCA 2.0** API. In Colab it installs the current `main` branch; in a local Jupyter session it uses the current project checkout. It keeps tensors and training internals out of sight and produces reusable result files.

### Before you begin

1. In Colab, choose **Runtime → Change runtime type → T4 GPU** (recommended). Local Jupyter sessions can use CPU or an available accelerator.
2. Run the cells from top to bottom.
3. Start with the included RNA example, or upload your own aligned FASTA/Stockholm file.

> Training a DCA model can take minutes to hours depending on alignment length, model type, and stopping limits. The default settings are intentionally modest for an interactive first run.


In [ ]:
# @title 1 · Install and initialize adabmDCA { display-mode: "form" }
# @markdown In Colab, installs the current `main` branch. In local Jupyter, uses this project checkout.

import importlib
import re
import shutil
import subprocess
import sys
from pathlib import Path

REPOSITORY = "https://github.com/spqb/adabmDCApy.git"
BRANCH = "main"
try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    files = None
    IN_COLAB = False

if IN_COLAB:
    REPO_DIR = Path("/content/adabmDCApy")
    print(f"Installing adabmDCA from {BRANCH!r} …")
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    subprocess.run(
        ["git", "clone", "--quiet", "--depth", "1", "--branch", BRANCH, REPOSITORY, str(REPO_DIR)],
        check=True,
    )
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet", "--upgrade", "-e", str(REPO_DIR)],
        check=True,
    )
else:
    candidates = (Path.cwd(), Path.cwd().parent)
    REPO_DIR = next(
        (path.resolve() for path in candidates if (path / "pyproject.toml").is_file() and (path / "adabmDCA").is_dir()),
        None,
    )
    if REPO_DIR is None:
        raise RuntimeError("Start Jupyter from the adabmDCApy repository or its notebooks directory.")
    print(f"Using local adabmDCA checkout at {REPO_DIR}")
# Editable installs use a .pth file that a running notebook kernel may not
# process until restart. Add the checkout explicitly so imports work now.
repo_path = str(REPO_DIR.resolve())
if repo_path not in sys.path:
    sys.path.insert(0, repo_path)
importlib.invalidate_caches()

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import HTML, Image, display

from adabmDCA import (
    AlignmentLoadConfig,
    TrainingConfig,
    load_alignment,
    load_model,
    normalize_gap_symbols,
    predict_contacts,
    read_alignment,
    sample_sequences,
    train_model,
)
from adabmDCA import __version__ as adabmdca_version
from adabmDCA.alphabet import detect_alphabet

commit = subprocess.run(
    ["git", "-C", str(REPO_DIR), "rev-parse", "--short", "HEAD"],
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()
device_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
runtime_source = f"{BRANCH}@{commit}" if IN_COLAB else f"local@{commit}"

display(HTML(
    f"<div style='padding:12px 16px;border-left:5px solid #16a34a;background:#f0fdf4'>"
    f"<b>Ready.</b> adabmDCA {adabmdca_version} · {runtime_source} · {device_name}</div>"
))


## Input and model

The next two cells define the alignment and either train a new model or load an existing parameter file. Invalid sequences and duplicates are reported before any expensive computation starts.


In [ ]:
# @title 2 · Choose the alignment { display-mode: "form" }
alignment_source = "Included RNA example" # @param ["Included RNA example", "Upload alignment", "Alignment file path"]
alignment_file = "" # @param {type:"string"}
alphabet = "auto" # @param ["auto", "protein", "rna", "dna", "custom"]
custom_alphabet = "-ACGU" # @param {type:"string"}
job_name = "DCA_model" # @param {type:"string"}
weights_mode = "Automatic reweighting" # @param ["Automatic reweighting", "Uniform weights", "Upload weights", "Weights file path"]
weights_file = "" # @param {type:"string"}

safe_job_name = re.sub(r"[^A-Za-z0-9_.-]+", "_", job_name.strip()).strip("._") or "DCA_model"
output_root = Path("/content") if IN_COLAB else REPO_DIR / "notebook_outputs"
output_dir = output_root / safe_job_name
output_dir.mkdir(parents=True, exist_ok=True)

def upload_one(label, destination):
    if files is None:
        raise RuntimeError("File uploads are available in Colab. In local Jupyter, select a file-path option.")
    print(label)
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Please upload exactly one file, then run the cell again.")
    name, content = next(iter(uploaded.items()))
    path = destination / Path(name).name
    path.write_bytes(content)
    return path

if alignment_source == "Included RNA example":
    alignment_path = REPO_DIR / "example_data" / "RF00379.fasta"
elif alignment_source == "Upload alignment":
    alignment_path = upload_one("Upload one aligned FASTA, FASTA.gz, or Stockholm file:", output_dir)
else:
    alignment_path = Path(alignment_file).expanduser()
    if not alignment_path.is_file():
        raise FileNotFoundError(f"Alignment file not found: {alignment_path}")

parsed_alignment = normalize_gap_symbols(read_alignment(alignment_path))
if alphabet == "auto":
    alphabet_value = detect_alphabet(parsed_alignment.sequences)
elif alphabet == "custom":
    alphabet_value = custom_alphabet.strip()
else:
    alphabet_value = alphabet
if not alphabet_value:
    raise ValueError("The custom alphabet cannot be empty.")

weights_path = None
no_reweighting = weights_mode == "Uniform weights"
if weights_mode == "Upload weights":
    weights_path = upload_one("Upload one text file containing one weight per input sequence:", output_dir)
elif weights_mode == "Weights file path":
    weights_path = Path(weights_file).expanduser()
    if not weights_path.is_file():
        raise FileNotFoundError(f"Weights file not found: {weights_path}")

loaded_alignment = load_alignment(
    parsed_alignment,
    config=AlignmentLoadConfig(
        alphabet=alphabet_value,
        invalid_sequences="drop",
        remove_duplicates=True,
    ),
)
report = loaded_alignment.to_dict()
display(pd.DataFrame({
    "input": [Path(alignment_path).name],
    "alphabet": [alphabet_value],
    "sequences kept": [report["retained_sequences"]],
    "length": [report["sequence_length"]],
    "invalid dropped": [len(report["dropped_indices"])],
    "duplicates dropped": [len(report["duplicate_indices"])],
    "states": [len(report["tokens"])],
}))


In [ ]:
# @title 3 · Train a model or upload parameters { display-mode: "form" }
model_action = "Train a new model" # @param ["Train a new model", "Upload parameter file", "Parameter file path"]
parameter_file = "" # @param {type:"string"}
model_type = "bmDCA" # @param ["bmDCA", "eaDCA", "edDCA", "edgeDCA"]

# @markdown **General training controls**
n_chains = 2000 # @param {type:"integer", min:10}
n_sweeps = 10 # @param {type:"integer", min:1}
max_steps = 500 # @param {type:"integer", min:1}
target_pearson = 0.80 # @param {type:"number", min:0, max:0.999}
learning_rate = 0.01 # @param {type:"number", min:0}
pseudocount = "auto" # @param ["auto", "0.95", "0.8", "0.1", "0.01", "0.001"]
sampler = "metropolis" # @param ["gibbs", "metropolis"]
training_dtype = "float32" # @param ["float32", "bfloat16", "float64"]
seed = 42 # @param {type:"integer"}

# @markdown **Sparse-model controls** (ignored when not applicable)
activation_steps = 10 # @param {type:"integer", min:1}
activation_fraction = 0.001 # @param {type:"number", min:0, max:1}
target_density = 0.50 # @param {type:"number", min:0, max:1}
decimation_rate = 0.01 # @param {type:"number", min:0, max:1}

if model_action == "Upload parameter file":
    params_path = upload_one("Upload one adabmDCA parameter file:", output_dir)
elif model_action == "Parameter file path":
    params_path = Path(parameter_file).expanduser()
    if not params_path.is_file():
        raise FileNotFoundError(f"Parameter file not found: {params_path}")

if model_action != "Train a new model":
    model = load_model(params_path, alphabet=alphabet_value, device="auto", dtype=training_dtype)
    training_result = None
    display(pd.DataFrame([model.metadata.to_dict()]))
else:
    resolved_pseudocount = None if pseudocount == "auto" else float(pseudocount)
    config = TrainingConfig(
        model_type=model_type,
        alphabet=alphabet_value,
        learning_rate=learning_rate,
        n_sweeps=n_sweeps,
        sampler=sampler,
        n_chains=n_chains,
        target_pearson=target_pearson,
        max_epochs=max_steps,
        pseudocount=resolved_pseudocount,
        seed=seed,
        no_reweighting=no_reweighting,
        activation_steps=activation_steps,
        activation_fraction=activation_fraction,
        target_density=target_density,
        decimation_rate=decimation_rate,
        device="auto",
        dtype=training_dtype,
        checkpoint_interval=max(1, min(50, max_steps)),
    )

    progress_box = display(HTML("<b>Preparing training…</b>"), display_id=True)
    def show_initialization(event):
        data = event.training
        progress_box.update(HTML(
            f"<div style='padding:10px 14px;background:#eff6ff;border-left:5px solid #2563eb'>"
            f"<b>Initialized {model_type}</b> · {data.retained_sequences:,} sequences · "
            f"L={data.sequence_length} · q={data.num_states} · M_eff={data.effective_sequences:.1f} · "
            f"{event.n_chains:,} chains on {event.device}</div>"
        ))

    def show_progress(event):
        pearson = event.metrics.get("Pearson")
        pearson_text = "—" if pearson is None else f"{pearson:.4f}"
        progress_box.update(HTML(
            f"<div style='padding:10px 14px;background:#eff6ff;border-left:5px solid #2563eb'>"
            f"<b>Training {model_type}</b> · {event.stage} · epoch {event.epoch} · "
            f"gradient {event.gradient_steps} · structure {event.structure_steps} · Pearson {pearson_text}</div>"
        ))

    training_result = train_model(
        alignment_path,
        config=config,
        weights_path=weights_path,
        output_dir=output_dir,
        label=safe_job_name,
        progress=show_progress,
        on_initialized=show_initialization,
    )
    model = training_result.model
    progress_box.update(HTML(
        f"<div style='padding:10px 14px;background:#f0fdf4;border-left:5px solid #16a34a'>"
        f"<b>Training finished.</b> {training_result.stop_reason} · "
        f"{training_result.gradient_steps} gradient steps · "
        f"{training_result.structure_steps} structure steps</div>"
    ))

    history = training_result.history_dataframe()
    display(history.tail(10))
    if "Pearson" in history and len(history):
        ax = history.plot(x="Epochs", y="Pearson", figsize=(7, 4), color="#2563eb", legend=False)
        ax.axhline(target_pearson, color="#dc2626", linestyle="--", label="target")
        ax.set(ylabel="Pearson correlation", title="Training progress")
        ax.legend()
        plt.show()


## Generate sequences

Sampling returns sequences, energies, and optional diagnostics in one structured result. The analysis option estimates the mixing time from the natural alignment, chooses the generation length, and saves autocorrelation, Pearson, Cij, and PCA plots (PC1–PC2 and PC3–PC4 with marginal histograms).


In [ ]:
# @title 4 · Generate sequences { display-mode: "form" }
num_sequences = 1000 # @param {type:"integer", min:1}
sampling_sweeps = 1000 # @param {type:"integer", min:1}
sampling_algorithm = "metropolis" # @param ["gibbs", "metropolis"]
analyze_sampling = True # @param {type:"boolean"}
mixing_multiplier = 2 # @param {type:"integer", min:1}
beta = 1.0 # @param {type:"number", min:0.001}
sampling_dtype = "float32" # @param ["float32", "bfloat16", "float64"]
sampling_seed = 42 # @param {type:"integer"}

sampling_box = display(HTML("<b>Preparing sampling…</b>"), display_id=True)
def show_sampling_progress(event):
    if event.completed == 1 or event.completed == event.total or event.completed % max(1, event.total // 20) == 0:
        sampling_box.update(HTML(
            f"<div style='padding:10px 14px;background:#f5f3ff;border-left:5px solid #7c3aed'>"
            f"<b>Sampling</b> · sweep {event.completed}/{event.total}</div>"
        ))

sampling_result = sample_sequences(
    model=model,
    n_sequences=num_sequences,
    n_sweeps=sampling_sweeps,
    sampler=sampling_algorithm,
    beta=beta,
    seed=sampling_seed,
    reference_fasta=alignment_path if analyze_sampling else None,
    weights_path=weights_path if analyze_sampling else None,
    no_reweighting=no_reweighting,
    mixing_multiplier=mixing_multiplier,
    dtype=sampling_dtype,
    collect_diagnostics=analyze_sampling,
    progress=show_sampling_progress,
)
sampling_artifacts = sampling_result.save_bundle(output_dir / "sampling", label=safe_job_name)
diagnostic_artifacts = {}
if analyze_sampling:
    diagnostic_artifacts = sampling_result.save_diagnostic_plots(
        output_dir / "sampling", label=safe_job_name
    )
    sampling_artifacts.update(diagnostic_artifacts)
sampling_box.update(HTML(
    f"<div style='padding:10px 14px;background:#f0fdf4;border-left:5px solid #16a34a'>"
    f"<b>Generated {len(sampling_result.sequences):,} sequences</b> in "
    f"{sampling_result.num_sweeps:,} sweeps.</div>"
))
display(sampling_result.to_dataframe().head(10))

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(sampling_result.energies, bins=40, color="#7c3aed", alpha=0.82, edgecolor="white")
ax.set(title="Generated-sequence energies", xlabel="DCA energy", ylabel="Count")
fig.tight_layout()
plt.show()

if analyze_sampling:
    display(pd.DataFrame({
        "diagnostic": list(diagnostic_artifacts),
        "file": [str(path) for path in diagnostic_artifacts.values()],
    }))
    display(Image(filename=str(sampling_artifacts["pca_1_2_plot"]), width=700))
    display(Image(filename=str(sampling_artifacts["pca_3_4_plot"]), width=700))


## Explore the model

These optional cells use the reusable model object returned above. They do not retrain or reload parameters.


In [ ]:
# @title 5 · Compare natural and generated energies { display-mode: "form" }
max_natural_sequences = 10000 # @param {type:"integer", min:1}

natural_sequences = loaded_alignment.alignment.sequences[:max_natural_sequences]
natural_scores = model.score_sequences(natural_sequences)
generated_scores = model.score_sequences(sampling_result.sequences)

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(natural_scores.energies, bins=50, density=True, alpha=0.65, label="Natural", color="#2563eb")
ax.hist(generated_scores.energies, bins=50, density=True, alpha=0.65, label="Generated", color="#7c3aed")
ax.set(title="DCA energy distributions", xlabel="DCA energy", ylabel="Density")
ax.legend()
plt.show()

energy_artifacts = natural_scores.save_bundle(output_dir / "analysis", stem="natural_energies")
display(pd.DataFrame({
    "set": ["Natural", "Generated"],
    "n": [len(natural_scores.energies), len(generated_scores.energies)],
    "mean energy": [natural_scores.energies.mean(), generated_scores.energies.mean()],
    "std. energy": [natural_scores.energies.std(), generated_scores.energies.std()],
}))


In [ ]:
# @title 6 · Predict contacts { display-mode: "form" }
contact_source = "Trained model" # @param ["Trained model", "Mean-field from alignment"]
contact_pseudocount = 0.5 # @param {type:"number", min:0, max:1}

if contact_source == "Trained model":
    contact_result = model.predict_contacts()
else:
    contact_result = predict_contacts(
        fasta_path=alignment_path,
        alphabet=alphabet_value,
        pseudocount=contact_pseudocount,
        device="auto",
    )
contact_artifacts = contact_result.save_bundle(output_dir / "contacts", label=safe_job_name)

fig, ax = plt.subplots(figsize=(7, 6))
image = ax.imshow(contact_result.scores, origin="lower", cmap="magma")
ax.set(title=f"Contact scores · {contact_result.method}", xlabel="Position j", ylabel="Position i")
fig.colorbar(image, ax=ax, label="APC-corrected score")
plt.show()


In [ ]:
# @title 7 · Scan all single-site mutations { display-mode: "form" }
# @markdown Leave the sequence blank to use the first retained natural sequence.
wild_type_sequence = "" # @param {type:"string"}
wild_type_name = "wild_type" # @param {type:"string"}

wild_type = wild_type_sequence.strip().upper() or loaded_alignment.alignment.sequences[0]
mutation_result = model.scan_mutations(wild_type, name=wild_type_name or "wild_type")
mutation_artifacts = mutation_result.save_bundle(output_dir / "mutations", stem=wild_type_name or "wild_type")
mutation_table = mutation_result.to_dataframe()
display(mutation_table.nsmallest(15, "delta_energy"))

heatmap = mutation_table.pivot(index="mutant", columns="position_1based", values="delta_energy")
fig, ax = plt.subplots(figsize=(max(8, model.metadata.length / 4), 5))
image = ax.imshow(heatmap, aspect="auto", cmap="coolwarm", interpolation="nearest")
ax.set(
    title="Single-mutation ΔE (lower is favored by the model)",
    xlabel="Sequence position (1-based)",
    ylabel="Mutant state",
    yticks=np.arange(len(heatmap.index)),
    yticklabels=heatmap.index,
)
fig.colorbar(image, ax=ax, label="ΔE")
plt.show()


In [ ]:
# @title 8 · Download all results { display-mode: "form" }
# @markdown Downloads the model checkpoints, tables, FASTA files, diagnostics, and figures saved above.

archive_path = Path(shutil.make_archive(str(output_dir), "zip", root_dir=output_dir))
print(f"Prepared {archive_path.name} ({archive_path.stat().st_size / 1e6:.1f} MB)")
if files is not None:
    files.download(str(archive_path))
else:
    display(HTML(f"<b>Archive ready:</b> <code>{archive_path.resolve()}</code>"))


---

### What changed in this version

- Colab installs the repository's **`main` branch**; local Jupyter uses the current checkout.
- Standard DNA, RNA, and protein alphabets can be detected automatically. Nonstandard data still requires an explicit custom alphabet.
- The notebook uses `TrainingConfig`, `train_model`, `load_model`, `sample_sequences`, reusable `DCAModel` methods, progress events, and structured result serializers.
- Core package internals (manual tensor encoding, parameter masks, samplers, checkpoint plumbing, and FASTA serialization) are no longer exposed to the notebook user.
- Sampling analysis saves the styled autocorrelation, Pearson, Cij, PC1–PC2, and PC3–PC4 figures. The PCA plots include marginal histograms.
- Inputs are validated early, filenames are sanitized, seeds are explicit, and all outputs are collected in one archive.
